# Project 2: Exploratory Data Analysis (EDA) Report
---
## Assigned Topic:
__[World Development Statistics]__

## Selected Problem Statement: 
__[Human Capital Strategy & Philanthropy]__

## 1. Introduction & Objectives
*(Guidelines: Introduce your topic and give a brief high-level overview of the industry or topic you are exploring so a non-technical audience understands the context.)*

### 1.1 Context & Background

### 1.2 Core Analytical Objectives
*(List the key targeted questions that will guide your analysis to solve the chosen problem statement. Use bullet points.)*
* ...
* ...

---

## 2. Environment Setup & Data Collection
*(Rubric Checkpoint: Ensure all modules are imported cleanly with appropriate standard aliases. Data files must be loaded using relative paths for replicability.)*

In [1]:
import numpy as np
import scipy.stats as stats
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import missingno as msno

In [2]:
gni=pd.read_csv('./data/gni_per_cap_atlas_method_con2021.csv')
life_exp=pd.read_csv('./data/life_expectancy.csv')
population=pd.read_csv('./data/population.csv')

---
## 3. Data Handling


In [3]:
# Initial assessment: Check shapes, null values, and summary data types
gni_long=pd.melt(frame=gni,id_vars='country',var_name='year',value_name='gni')
life_exp_long=pd.melt(frame=life_exp,id_vars='country',var_name='year',value_name='life_exp')
population_long=pd.melt(frame=population,id_vars='country',var_name='year',value_name='population')


In [4]:
print(gni_long.shape)
print(life_exp_long.shape)
print(population_long.shape)

(47941, 3)
(58695, 3)
(59297, 3)


In [5]:
print(gni_long.isna().sum())
print(life_exp_long.isna().sum())
print(population_long.isna().sum())

country      0
year         0
gni        229
dtype: int64
country        0
year           0
life_exp    2079
dtype: int64
country         0
year            0
population    100
dtype: int64


In [6]:
print(gni_long.dtypes)
print(life_exp_long.dtypes)
print(population_long.dtypes)

country       str
year          str
gni        object
dtype: object
country         str
year            str
life_exp    float64
dtype: object
country       str
year          str
population    str
dtype: object


In [7]:
gni_countries = set( gni_long['country'] )
life_countries = set( life_exp_long['country'] )
pop_countries = set( population_long['country'] )

missing_in_gni_but_in_pop = pop_countries - gni_countries
missing_in_life_but_in_pop = pop_countries - life_countries

print("In Population but missing in GNI:", missing_in_gni_but_in_pop)
print("In Population but missing in Life:", missing_in_life_but_in_pop)

In Population but missing in GNI: {'Andorra', 'San Marino', 'Taiwan', 'Monaco', 'North Korea', 'Holy See'}
In Population but missing in Life: {'Holy See', 'Liechtenstein'}


In [8]:
# Data cleaning operations (handling data types, text styling, missing values, etc.)



**converting the years to integres**

In [9]:
population_long['year']=population_long['year'].astype(int)
gni_long['year']=gni_long['year'].astype(int)
life_exp_long['year']=life_exp_long['year'].astype(int)

**Creating functions to covert the gni and populations into numbers**

In [10]:
def convert_gni_num(x):
    
    value= str(x['gni'])
    
    if 'M' in value:
        y=value.strip('M')
        return float(y)*1000000
    elif 'k' in value:
        y=value.strip('k')
        return float(y)*1000
    else:
        return float(value)

def convert_pop_num(x):
    
    value= str(x['population'])
    
    if 'B' in value:
        y=value.strip('B')
        return float(y)*1000000000
    elif 'M' in value:
        y=value.strip('M')
        return float(y)*1000000
    elif 'k' in value:
        y=value.strip('k')
        return float(y)*1000
    else:
        return float(value)

In [11]:
population_long['population']=population_long.apply(convert_pop_num,axis=1)
gni_long['gni']=gni_long.apply(convert_gni_num,axis=1)

**Handiling tables from any Nan values**

In [12]:
population_long[(population_long['population'].isna())].sort_values('year').head()

,country,year,population
39671,Holy See,2001,NaN
39868,Holy See,2002,NaN
40065,Holy See,2003,NaN
40262,Holy See,2004,NaN
40459,Holy See,2005,NaN


**Noticing that population null values are all missing values from 2001 --> 2100 from Holy See**

In [13]:
life_exp_long[(life_exp_long['life_exp'].isna())].sort_values('year')
exluded_life_countries=['Andorra','Dominica','St. Kitts and Nevis','Monaco','Marshall Islands','Nauru','Palau','San Marino','Tuvalu']

**Noticing that there are (9) countries are repeated and have null life expextance values,\
except for years from 1950 unti 2019**

**Double checking there are no missing values athor from the exluded countries:**

In [14]:
life_exp_long[(~life_exp_long['country'].isin(exluded_life_countries)) & (life_exp_long['life_exp'].isna())]

,country,year,life_exp


In [15]:
gni_long[(gni_long['gni'].isna())].sort_values('year').head()

,country,year,gni
100,Liechtenstein,1800,NaN
291,Liechtenstein,1801,NaN
482,Liechtenstein,1802,NaN
673,Liechtenstein,1803,NaN
864,Liechtenstein,1804,NaN


**Noticing that Liechtenstein always had nulll values in the GNI**

- So, I conclude that it is safe to drop the null values in population, life expectence and gni tables. 

In [16]:
# cleaned_population=population_long.dropna()
# cleaned_gni=gni_long.dropna()
# cleaned_life_exp=life_exp_long.dropna()

# cleaned_pop_life=pd.merge(left=cleaned_population, 
#                   right=cleaned_life_exp,
#                   how='left',
#                   on=(['country','year']))
# cleaned_pop_life_gni=pd.merge(left=cleaned_pop_life,
#                       right=cleaned_gni,
#                       how='left',
#                       on=(['country','year']))

In [17]:
pop_life=pd.merge(left=population_long, 
                  right=life_exp_long,
                  how='left',
                  on=(['country','year']))
pop_life_gni=pd.merge(left=pop_life,
                      right=gni_long,
                      how='left',
                      on=(['country','year']))

In [18]:
pop_life_gni.isna().sum()

country           0
year              0
population      100
life_exp       2681
gni           11585
dtype: int64

---
## 4. Exploratory Data Analysis (EDA)
*(Rubric Checkpoint: Demonstrate mastery of basic descriptive statistics, sorting records, and applying Boolean masking techniques to isolate specific baseline metrics.)*

### 4.1. ----

In [19]:
cleaned_pop_life_gni=pop_life_gni.dropna(subset=['gni', 'life_exp', 'population'])

**Dropping 6 counties scince there data collection just from 1950 until 2019\
which causes unfair comparision**

In [20]:
exluded_countries=['Dominica','St. Kitts and Nevis','Marshall Islands','Nauru','Palau','Tuvalu']
cleaned_pop_life_gni=cleaned_pop_life_gni[~cleaned_pop_life_gni.isin(exluded_countries)]

**seeining the percentange of our data after filteration** 

In [21]:
print(f"Original data set size: {len(pop_life_gni)}")
print(f"After Drop data set size: {len(cleaned_pop_life_gni)}")
print(f"Percentage of data retained: {len(cleaned_pop_life_gni)/len(pop_life_gni)*100:.1f}%")

Original data set size: 59297
After Drop data set size: 46604
Percentage of data retained: 78.6%


**Summarizing the data**

In [24]:

summary=pd.DataFrame()
summary['year_start'] = cleaned_pop_life_gni.groupby('country')['year'].first()
summary['year_end'] = cleaned_pop_life_gni.groupby('country')['year'].last()
summary['population_start'] = cleaned_pop_life_gni.groupby('country')['population'].first()
summary['population_end'] = cleaned_pop_life_gni.groupby('country')['population'].last()
summary['life_exp_start'] = cleaned_pop_life_gni.groupby('country')['life_exp'].first()
summary['life_exp_end'] = cleaned_pop_life_gni.groupby('country')['life_exp'].last()
summary['gni_start'] = cleaned_pop_life_gni.groupby('country')['gni'].first()
summary['gni_end'] = cleaned_pop_life_gni.groupby('country')['gni'].last()
summary=summary.reset_index()
summary.head()

,country,year_start,year_end,population_start,population_end,life_exp_start,life_exp_end,gni_start,gni_end
0,Afghanistan,1800,2050,3280000.0,74100000.0,28.2,70.0,207.0,907.0
1,Albania,1800,2050,400000.0,2460000.0,35.4,82.8,207.0,11600.0
2,Algeria,1800,2050,2500000.0,60000000.0,28.8,81.2,446.0,5370.0
3,Angola,1800,2050,1570000.0,72300000.0,27.0,73.1,517.0,3340.0
4,Antigua and Barbuda,1800,2050,37000.0,99000.0,33.5,80.2,650.0,26900.0


### 4.2. ----

In [25]:
# Target Analysis Block 1:
# 1. Set your strategic boundaries
LOW_GNI_LIMIT = 4500
LOW_LIFE_LIMIT = 60

HIGH_GNI_TARGET = 14000
HIGH_LIFE_TARGET = 75

# 2. Create the 4-part logical condition (The Mask)
# Change the column names inside the brackets if yours are named slightly differently!
ymasky = (
    (summary['gni_start'] < LOW_GNI_LIMIT) & 
    (summary['life_exp_start'] < LOW_LIFE_LIMIT) & 
    (summary['gni_end'] > HIGH_GNI_TARGET) & 
    (summary['life_exp_end'] > HIGH_LIFE_TARGET)
)

# 3. Apply the mask to extract your Blueprint Nations
yblueprint_countriesy = summary[ymasky]['country'].tolist()

print(f"Found {len(yblueprint_countriesy)} Blueprint Nations:")
print(yblueprint_countriesy)

Found 74 Blueprint Nations:
['Antigua and Barbuda', 'Argentina', 'Australia', 'Austria', 'Bahamas', 'Bahrain', 'Barbados', 'Belgium', 'Brazil', 'Brunei', 'Bulgaria', 'Canada', 'Chile', 'China', 'Costa Rica', 'Croatia', 'Cuba', 'Cyprus', 'Czech Republic', 'Denmark', 'Dominican Republic', 'Estonia', 'Finland', 'France', 'Germany', 'Greece', 'Grenada', 'Hong Kong, China', 'Hungary', 'Iceland', 'Ireland', 'Israel', 'Italy', 'Japan', 'Kazakhstan', 'Kuwait', 'Latvia', 'Libya', 'Lithuania', 'Luxembourg', 'Malaysia', 'Maldives', 'Malta', 'Mauritius', 'Mexico', 'Montenegro', 'New Zealand', 'Norway', 'Oman', 'Panama', 'Poland', 'Portugal', 'Qatar', 'Romania', 'Russia', 'Saudi Arabia', 'Serbia', 'Seychelles', 'Singapore', 'Slovak Republic', 'Slovenia', 'South Korea', 'Spain', 'St. Lucia', 'St. Vincent and the Grenadines', 'Sweden', 'Switzerland', 'Thailand', 'Trinidad and Tobago', 'Turkey', 'United Arab Emirates', 'United Kingdom', 'United States', 'Uruguay']


### 4.3. ----

In [ ]:
# Visualization 1 

**Visualization 1 Interpretation:**
> *Write 2-3 sentences explaining exactly what relationship or pattern this chart demonstrates and how it addresses your project objectives.*

In [ ]:
# Visualization 2

**Visualization 2 Interpretation:**
> *Write 2-3 sentences explaining exactly what relationship or pattern this chart demonstrates and how it addresses your project objectives.*

---
## 5. Summary of Findings & Actionable Recommendations
*(Guidelines: Connect individual observations into a cohesive narrative that guides the target non-technical audience toward data-driven actions.)*

### 5.1 Key Insights (Summary of Findings)
*(Use bullet points to highlight the core truths revealed by your analysis)*
* ...
* ...

### 5.2 Actionable Recommendations
*(Translate your data into clear choices or recommendations for your stakeholder based directly on the problem statement)*
* ...
* ...

### 5.3 Limitations & Areas for Further Research
* ...

---

## 6. Data Dictionary & References

### 6.1 Data Dictionary
| Feature / Column | Data Type | Source | Description |
|:---|:---|:---|:---|
| column_name_1 | type | Original / Engineered | Detailed explanation of what this variable represents |
| column_name_2 | type | Original / Engineered | Detailed explanation of what this variable represents |

### 6.2 References & Sources
* *Source 1:*